### 문항 1 네이버 VIBE Top 100 수집

In [ ]:
# 대상: https://vibe.naver.com/chart

# 추출 필드: 순위 / 곡명 / 아티스트

# 아티스트는 리스트로 담을 것 (협업곡은 여러 명)

# Selenium 사용 금지

# 100건이 모두 수집되었는지 건수로 확인할 것

# 결과를 vibe_top100.csv로 저장할 것

In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [10]:
URL = 'https://vibe.naver.com/chart'

API = 'https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total'

PARAMS = {'start': 1, 'display': 100}

HEADERS = {'User-Agent':'Mozilla 5.0','Accept':'application/json'}

res = requests.get(API, params={**PARAMS}, headers=HEADERS)

res.status_code


200

In [ ]:
res.json()['response']['result']['chart']['items']['tracks']

In [ ]:
items = res.json()['response']['result']['chart']['items']['tracks']
chart = []
for item in items:
    chart.append({
        '순위': item['rank']['currentRank'],
        '곡명': item['trackTitle'],
        '아티스트': [p.get('artistName', '') for p in item['artists']],
    })
chart

In [ ]:
## 최적화
URL = 'https://vibe.naver.com/chart'

API = 'https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total'

PARAMS = {'start': 1, 'display': 100}

HEADERS = {'User-Agent':'Mozilla 5.0','Accept':'application/json'}

def fetch_vibe_chart():
    res = requests.get(API, params={**PARAMS}, headers=HEADERS)
    res.raise_for_status
    items = res.json()['response']['result']['chart']['items']['tracks']
    chart = []
    for item in items:
        chart.append({
            '순위': item['rank']['currentRank'],
            '곡명': item['trackTitle'],
            '아티스트': [p.get('artistName', '') for p in item['artists']],
        })
    return chart

chart = fetch_vibe_chart()
print(f'{len(chart)}건 수집 완료')
df = pd.DataFrame(chart)
df.to_csv('vibe_top100.csv', index=False, encoding='utf-8-sig')


100건 수집 완료


### 문항 2 삼성전자 일별 시세 1년치 수집

In [ ]:
# 대상: https://finance.naver.com/item/sise.naver?code=005930

# 추출 필드: 날짜 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량

# 1년치 전량을 페이지네이션으로 수집할 것

# 숫자는 정제하여 숫자로 변환할 것

# 마지막 페이지를 자동으로 감지해 종료할 것

# 결과를 samsung_1y.csv로 저장할 것

In [19]:
URL = 'https://finance.naver.com/item/sise_day.naver'

HEADERS = {'User-Agent':'Mozilla 5.0'}

res = requests.get(URL, params={'code':'005930','page':2}, headers=HEADERS)
res.status_code

200

In [28]:
soup = BeautifulSoup(res.text, 'html.parser')
row = []
trs = soup.select('table.type2 tr')
for tr in trs:
    td = tr.select('td')
    if len(td) < 7 :
        continue
    plus_mius = td[2].select_one('em.bu_p').text.strip()
    amount = int((td[2].select_one('span.tah').text.strip()).replace(',',''))
    row.append({
        '날짜': td[0].text.strip(),
        '종가': int(td[1].text.strip().replace(',','')),
        '전일비':f'{plus_mius} {amount}',
        '시가': int(td[3].text.strip().replace(',','')),
        '고가':int(td[4].text.strip().replace(',','')),
        '저가':int(td[5].text.strip().replace(',','')),
        '거래량':int(td[6].text.strip().replace(',','')),
    })
row



[{'날짜': '2026.08.06',
  '종가': 230500,
  '전일비': '하락 15500',
  '시가': 241500,
  '고가': 246000,
  '저가': 228000,
  '거래량': 26101823},
 {'날짜': '2026.08.05',
  '종가': 246000,
  '전일비': '상승 6000',
  '시가': 254000,
  '고가': 254500,
  '저가': 244000,
  '거래량': 22577128},
 {'날짜': '2026.08.04',
  '종가': 240000,
  '전일비': '상승 500',
  '시가': 244500,
  '고가': 244500,
  '저가': 228000,
  '거래량': 29433821},
 {'날짜': '2026.08.03',
  '종가': 239500,
  '전일비': '하락 23000',
  '시가': 248000,
  '고가': 249500,
  '저가': 238000,
  '거래량': 27825493},
 {'날짜': '2026.07.31',
  '종가': 262500,
  '전일비': '상승 55500',
  '시가': 257000,
  '고가': 267000,
  '저가': 243000,
  '거래량': 58478873},
 {'날짜': '2026.07.30',
  '종가': 207000,
  '전일비': '하락 1500',
  '시가': 214000,
  '고가': 226000,
  '저가': 202000,
  '거래량': 46694193},
 {'날짜': '2026.07.29',
  '종가': 208500,
  '전일비': '하락 11500',
  '시가': 226500,
  '고가': 234000,
  '저가': 189200,
  '거래량': 65555523},
 {'날짜': '2026.07.28',
  '종가': 220000,
  '전일비': '하락 34000',
  '시가': 238500,
  '고가': 240000,
  '저가': 218500,
  '거래량':

In [34]:
## 최적화 (숫자 변환 함수도 필요)
import re 
from datetime import datetime, timedelta

URL = 'https://finance.naver.com/item/sise_day.naver'

HEADERS = {'User-Agent':'Mozilla 5.0'}

def get_num(a):
    m = re.search(r'\d+', a.replace(',','').strip())
    return int(m.group()) if m else ''

def fetch(page):
    res = requests.get(URL, params={'code':'005930','page':page}, headers=HEADERS)
    res.raise_for_status
    return res.text

def parse(html):
    soup = BeautifulSoup(html, 'html.parser')
    row = []
    trs = soup.select('table.type2 tr')
    for tr in trs:
        td = tr.select('td')
        if len(td) < 7 :
            continue
        plus_mius = td[2].select_one('em.bu_p').text.strip()
        amount = get_num(td[2].select_one('span.tah').text)
        row.append({
            '날짜': td[0].text.strip(),
            '종가': get_num(td[1].text),
            '전일비':f'{plus_mius} {amount}',
            '시가': get_num(td[3].text),
            '고가':get_num(td[4].text),
            '저가':get_num(td[5].text),
            '거래량':get_num(td[6].text),
        })
    return row

result = []

for page in range(1, 26):
    rows = parse(fetch(page))

    
    if not rows:
        print(f"{page}페이지가 비어 있습니다. 종료.")
        break

    
    last_date = rows[0]['날짜'] 
    if datetime.strptime(last_date, '%Y.%m.%d') < datetime.now() - timedelta(days=365):
        print(f"{last_date}, 1년 이전 날짜에 도달했습니다. 종료.")
        break

    result.extend(rows)
    print(f"{page}페이지 · 누적 {len(result)}건 · 최종 {rows[-1]['날짜']}")
    time.sleep(0.5)

df = pd.DataFrame(result).drop_duplicates(subset=["날짜"])
df.to_csv("samsung_1y.csv", index=False, encoding="utf-8-sig")
print(f"\n최종 {len(df)}건 · {df['날짜'].min()} ~ {df['날짜'].max()}")


1페이지 · 누적 10건 · 최종 2026.08.07
2페이지 · 누적 20건 · 최종 2026.07.24
3페이지 · 누적 30건 · 최종 2026.07.09
4페이지 · 누적 40건 · 최종 2026.06.25
5페이지 · 누적 50건 · 최종 2026.06.11
6페이지 · 누적 60건 · 최종 2026.05.27
7페이지 · 누적 70건 · 최종 2026.05.12
8페이지 · 누적 80건 · 최종 2026.04.24
9페이지 · 누적 90건 · 최종 2026.04.10
10페이지 · 누적 100건 · 최종 2026.03.27
11페이지 · 누적 110건 · 최종 2026.03.13
12페이지 · 누적 120건 · 최종 2026.02.26
13페이지 · 누적 130건 · 최종 2026.02.09
14페이지 · 누적 140건 · 최종 2026.01.26
15페이지 · 누적 150건 · 최종 2026.01.12
16페이지 · 누적 160건 · 최종 2025.12.24
17페이지 · 누적 170건 · 최종 2025.12.10
18페이지 · 누적 180건 · 최종 2025.11.26
19페이지 · 누적 190건 · 최종 2025.11.12
20페이지 · 누적 200건 · 최종 2025.10.29
21페이지 · 누적 210건 · 최종 2025.10.15
22페이지 · 누적 220건 · 최종 2025.09.24
23페이지 · 누적 230건 · 최종 2025.09.10
24페이지 · 누적 240건 · 최종 2025.08.27
25페이지 · 누적 250건 · 최종 2025.08.12

최종 250건 · 2025.08.12 ~ 2026.08.21
